# Modelo no supervisado — Riesgo sísmico en Santander
**Hackathon CIAI UTSmart 2026** · Componente *no supervisado* (§3 Medio, requerido también en Avanzado)

**Pregunta:** ¿qué unidades territoriales de Santander (municipios o celdas) comparten un comportamiento sísmico parecido y cuáles se parecen al perfil del Nido Sísmico de Bucaramanga?

**Método:** se agrega el catálogo del SGC por unidad territorial → ~20 variables → estandarización → comparación de K-Means, Clustering Jerárquico (Ward) y Mezcla Gaussiana (GMM) con métricas internas y estabilidad → interpretación de grupos, importancia de variables y mapa.

**Fuente:** catálogo sísmico RSN-SGC (http://bdrsnc.sgc.gov.co/paginas1/catalogo/). Anotar fecha de descarga y filtros en `entregables/fuente_oficial.md`.

## 0. Configuración (lo único que probablemente tengas que tocar)

In [ ]:
import re, glob, unicodedata, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             calinski_harabasz_score, adjusted_rand_score)
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# --- Datos ---------------------------------------------------------------
RUTA_CATALOGOS = "datos/raw/*"        # .xlsx / .xls / .csv del SGC (puedes poner varios: ambos periodos)
FECHA_MIN, FECHA_MAX = "2006-01-01", "2026-10-13"   # últimos ~20 años
MAG_MIN = 0.0                          # sube a 1.5 o 2.0 si quieres evitar sesgo por detección de sismos pequeños
# Santander (caja de coordenadas aproximada). Ajústala a tu zona de estudio.
LAT_RANGO, LON_RANGO = (5.7, 8.2), (-74.6, -72.4)

# --- Unidad territorial --------------------------------------------------
UNIDAD = "municipio"                   # "municipio" o "celda" (rejilla de 0.1° ≈ 11 km)
TAM_CELDA = 0.1
MIN_EVENTOS = 15                       # unidades con menos sismos se descartan (estadísticas poco fiables)

# --- Referencia territorial: centro del Nido Sísmico de Bucaramanga (aprox.) ---
NIDO_LAT, NIDO_LON = 6.8, -73.1

# --- Variables territoriales externas (DANE/IGAC), OPCIONAL ----------------
# CSV con columna "municipio" + las que tengas (poblacion, altitud_msnm, viviendas...). None si no hay.
RUTA_TERRITORIAL = None                # ej: "datos/raw/dane_municipios.csv"

## 1. Carga y unificación de los catálogos
El SGC entrega dos consultas con columnas distintas (1993–feb 2018 y desde mar 2018). Esta celda detecta la fila de encabezado, normaliza los nombres de columnas por palabras clave y unifica todo en una sola tabla. **Revisa el diccionario de columnas detectadas que imprime**: si alguna quedó mal, corrígela en `MAPA_MANUAL`.

In [ ]:
def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]+", " ", s).strip()

# palabra clave -> nombre estándar (se evalúa en orden; la primera que coincide gana)
REGLAS = [
    ("error", None),                       # ignora columnas de error (lat/lon/prof)
    ("fecha", "fecha"), ("hora", "hora"),
    ("latitud", "lat"), ("longitud", "lon"),
    ("profundidad", "prof"),
    ("tipo", "tipo_mag"),
    ("magnitud", "mag"),
    ("rms", "rms"), ("gap", "gap"),
    ("fases", "n_fases"),
    ("municipio", "municipio"), ("departamento", "depto"),
]
MAPA_MANUAL = {}   # ej: {"Magnitud Mw": "mag"}  para forzar un mapeo

def leer_catalogo(ruta):
    if ruta.lower().endswith((".xlsx", ".xls")):
        crudo = pd.read_excel(ruta, header=None)
    else:
        crudo = pd.read_csv(ruta, header=None, sep=None, engine="python")
    # fila de encabezado = primera que contiene "latitud"
    fila = next(i for i in range(min(30, len(crudo)))
                if crudo.iloc[i].astype(str).map(norm).str.contains("latitud").any())
    df = crudo.iloc[fila + 1:].copy()
    df.columns = crudo.iloc[fila].astype(str).str.strip()
    return df.reset_index(drop=True)

def estandarizar_columnas(df):
    mapa, usados = {}, set()
    for col in df.columns:
        if col in MAPA_MANUAL:
            mapa[col] = MAPA_MANUAL[col]; usados.add(MAPA_MANUAL[col]); continue
        n = norm(col)
        for clave, destino in REGLAS:
            if clave in n:
                if destino and destino not in usados:
                    mapa[col] = destino; usados.add(destino)
                break
    return df[list(mapa)].rename(columns=mapa), mapa

partes = []
for ruta in sorted(glob.glob(RUTA_CATALOGOS)):
    if not ruta.lower().endswith((".xlsx", ".xls", ".csv")):
        continue
    d, mapa = estandarizar_columnas(leer_catalogo(ruta))
    print(f"{ruta}: {len(d):,} filas | columnas -> {mapa}")
    partes.append(d)
assert partes, "No encontré archivos en RUTA_CATALOGOS"
raw = pd.concat(partes, ignore_index=True, sort=False)
print("\nTotal bruto:", f"{len(raw):,}", "| columnas:", list(raw.columns))

## 2. Limpieza
Duplicados, tipos, fechas homogéneas, faltantes, filtro de zona/periodo y revisión de atípicos.

In [ ]:
df = raw.copy()

# fecha (+ hora si viene separada)
if "hora" in df and "fecha" in df:
    df["fecha"] = pd.to_datetime(df["fecha"].astype(str).str[:10] + " " + df["hora"].astype(str), errors="coerce")
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")

for c in ["lat", "lon", "prof", "mag", "rms", "gap", "n_fases"]:
    if c in df: df[c] = pd.to_numeric(df[c], errors="coerce")

n0 = len(df)
df = df.drop_duplicates()
df = df.dropna(subset=["fecha", "lat", "lon", "mag"])              # sin coordenadas o magnitud -> fuera
df = df[(df.fecha >= FECHA_MIN) & (df.fecha <= FECHA_MAX)]
df = df[df.lat.between(*LAT_RANGO) & df.lon.between(*LON_RANGO) & (df.mag >= MAG_MIN)]
df = df[df.prof.isna() | (df.prof >= 0)]                           # profundidades negativas = error
print(f"Filas: {n0:,} -> {len(df):,} tras limpieza")

# faltantes en variables de calidad: mediana (se documenta en el diccionario de datos)
for c in ["prof", "rms", "gap", "n_fases"]:
    if c in df:
        print(f"  {c}: {df[c].isna().mean():.1%} faltantes -> mediana")
        df[c] = df[c].fillna(df[c].median())
    else:
        df[c] = np.nan
        print(f"  {c}: columna no existe en el catálogo (se omitirá)")

# unidad territorial
if UNIDAD == "municipio" and "municipio" in df:
    df["unidad"] = df["municipio"].astype(str).str.upper().str.strip()
else:
    if UNIDAD == "municipio":
        print("Sin columna municipio -> uso celdas")
    df["unidad"] = ("C" + (np.floor(df.lat / TAM_CELDA) * TAM_CELDA).round(2).astype(str)
                    + "_" + (np.floor(df.lon / TAM_CELDA) * TAM_CELDA).round(2).astype(str))
df["anio"] = df.fecha.dt.year
display(df.describe().T.round(2))

## 3. Ingeniería de variables por unidad territorial

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

df["energia"] = 10 ** (1.5 * df.mag + 4.8)          # Julios (Gutenberg-Richter)
df["es_m3"] = (df.mag >= 3.0).astype(int)
df["es_m4"] = (df.mag >= 4.0).astype(int)
df["sup"] = (df.prof < 30).astype(int)               # superficial
df["int"] = df.prof.between(30, 150, inclusive="left").astype(int)   # intermedia
df["pro"] = (df.prof >= 150).astype(int)             # profunda (Nido)
df["reciente"] = (df.fecha >= df.fecha.max() - pd.DateOffset(years=5)).astype(int)
anios_tot = max((df.fecha.max() - df.fecha.min()).days / 365.25, 1)

g = df.groupby("unidad")
F = pd.DataFrame({
    "n_sismos":        g.size(),
    "sismos_x_anio":   g.size() / anios_tot,
    "mag_media":       g.mag.mean(),
    "mag_max":         g.mag.max(),
    "mag_std":         g.mag.std(),
    "mag_p90":         g.mag.quantile(0.90),
    "pct_m3_mas":      g.es_m3.mean(),
    "pct_m4_mas":      g.es_m4.mean(),
    "log_energia":     np.log10(g.energia.sum()),
    "prof_media":      g.prof.mean(),
    "prof_std":        g.prof.std(),
    "pct_superficial": g.sup.mean(),
    "pct_intermedia":  g["int"].mean(),
    "pct_profunda":    g.pro.mean(),
    "rms_medio":       g.rms.mean(),
    "gap_medio":       g.gap.mean(),
    "fases_medias":    g.n_fases.mean(),
    "frac_reciente":   g.reciente.mean(),                      # tendencia: ¿qué tanto de su sismicidad es de los últimos 5 años?
    "anios_activos":   g.anio.nunique() / anios_tot,           # persistencia temporal
    "lat_centro":      g.lat.mean(),                           # variables territoriales
    "lon_centro":      g.lon.mean(),
})
F["dist_nido_km"] = haversine(F.lat_centro, F.lon_centro, NIDO_LAT, NIDO_LON)

# variables territoriales externas (DANE/IGAC) opcionales
if RUTA_TERRITORIAL:
    T = pd.read_csv(RUTA_TERRITORIAL)
    T["municipio"] = T["municipio"].astype(str).str.upper().str.strip()
    F = F.join(T.set_index("municipio"), how="left")
    F = F.fillna(F.median(numeric_only=True))

F = F.dropna(axis=1, how="all")                                 # quita variables que no existían en el catálogo
F = F[F.n_sismos >= MIN_EVENTOS].copy()
F = F.fillna(F.median(numeric_only=True))
print(f"{len(F)} unidades con >= {MIN_EVENTOS} sismos | {F.shape[1]} variables")
assert F.shape[1] >= 15, "Faltan variables: el reto Medio exige >= 15"
display(F.round(2).head(10))

### Diccionario de variables (pásalo a `entregables/diccionario_de_datos.md`)
| Variable | Unidad | Por qué se incluye |
|---|---|---|
| n_sismos, sismos_x_anio | conteo, 1/año | Frecuencia: identifica hotspots |
| mag_media, mag_max, mag_std, mag_p90 | Ml/Mw | Tamaño típico y extremos de los eventos |
| pct_m3_mas, pct_m4_mas | fracción | Proporción de eventos potencialmente sentidos / dañinos |
| log_energia | log10 J | Energía liberada acumulada (domina por los eventos grandes) |
| prof_media, prof_std, pct_superficial/intermedia/profunda | km, fracción | Los sismos superficiales son más dañinos por menor atenuación; el Nido es intermedio-profundo |
| rms_medio, gap_medio, fases_medias | s, °, conteo | Calidad de la localización: separa señal real de incertidumbre |
| frac_reciente, anios_activos | fracción | Tendencia y persistencia temporal |
| lat_centro, lon_centro, dist_nido_km | °, km | Variables territoriales (posición y distancia al Nido de Bucaramanga) |

## 4. Análisis exploratorio

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.3))
df.groupby("anio").size().plot(kind="bar", ax=ax[0], color="#c0392b"); ax[0].set_title("Sismos por año")
df.mag.plot(kind="hist", bins=40, ax=ax[1], color="#2980b9"); ax[1].set_title("Histograma de magnitudes"); ax[1].set_yscale("log")
sc = ax[2].scatter(df.lon, df.lat, c=df.prof, s=3, cmap="viridis_r", alpha=.5)
plt.colorbar(sc, ax=ax[2], label="Profundidad (km)"); ax[2].set_title("Epicentros (lon-lat)")
plt.tight_layout(); plt.show()

# correlaciones fuertes -> redundancia
corr = F.drop(columns=["lat_centro", "lon_centro"]).corr()
fig, ax = plt.subplots(figsize=(9, 7)); im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=7)
plt.colorbar(im); ax.set_title("Correlación entre variables"); plt.tight_layout(); plt.show()

## 5. Preprocesamiento
Las variables de conteo y energía tienen colas muy largas (el Nido pesa órdenes de magnitud más que un municipio tranquilo), así que se aplica `log1p` a las de escala y luego **RobustScaler** (mediana/IQR) para que unos pocos municipios extremos no aplasten al resto.

In [ ]:
COLS_LOG = [c for c in ["n_sismos", "sismos_x_anio", "dist_nido_km", "poblacion", "viviendas"] if c in F]
X = F.copy()
X[COLS_LOG] = np.log1p(X[COLS_LOG])
# las coordenadas se usan para el mapa y la distancia al Nido, no para agrupar (evita grupos puramente geográficos)
COLS_MODELO = [c for c in X.columns if c not in ("lat_centro", "lon_centro")]
Xs = RobustScaler().fit_transform(X[COLS_MODELO])
Xs = np.clip(Xs, -5, 5)                         # recorte final de atípicos extremos
print("Variables que entran al modelo:", len(COLS_MODELO))

pca = PCA(random_state=RANDOM_STATE).fit(Xs)
var_acum = pca.explained_variance_ratio_.cumsum()
n_pc = int(np.argmax(var_acum >= 0.90) + 1)
print(f"{n_pc} componentes explican el 90% de la varianza")
Z = PCA(n_components=n_pc, random_state=RANDOM_STATE).fit_transform(Xs)   # espacio de agrupamiento (reduce ruido y colinealidad)
Z2 = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(Xs)     # solo para dibujar

## 6. Selección de modelo y de *k*
Se comparan **tres enfoques** (exigido por la rúbrica: *comparación de enfoques*) para k = 2…8 con tres métricas internas:
- **Silhouette** ↑ (cohesión vs. separación) · **Davies-Bouldin** ↓ · **Calinski-Harabasz** ↑
- **Estabilidad**: ARI promedio entre el agrupamiento completo y el de submuestras del 80%.

In [ ]:
def ajustar(nombre, k, X):
    if nombre == "KMeans":
        return KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE).fit_predict(X)
    if nombre == "Ward":
        return AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(X)
    if nombre == "GMM":
        return GaussianMixture(n_components=k, covariance_type="diag", n_init=5, random_state=RANDOM_STATE).fit_predict(X)

def estabilidad(nombre, k, X, n=20, frac=0.8):
    rng = np.random.default_rng(RANDOM_STATE); base = ajustar(nombre, k, X); aris = []
    for _ in range(n):
        idx = rng.choice(len(X), int(frac * len(X)), replace=False)
        aris.append(adjusted_rand_score(base[idx], ajustar(nombre, k, X[idx])))
    return np.mean(aris)

filas = []
for nombre in ["KMeans", "Ward", "GMM"]:
    for k in range(2, min(9, len(Z) // 3)):
        lab = ajustar(nombre, k, Z)
        if len(set(lab)) < 2: continue
        filas.append(dict(modelo=nombre, k=k,
                          silhouette=silhouette_score(Z, lab),
                          davies_bouldin=davies_bouldin_score(Z, lab),
                          calinski_harabasz=calinski_harabasz_score(Z, lab),
                          estabilidad_ARI=estabilidad(nombre, k, Z),
                          min_tam_grupo=np.bincount(lab).min()))
R = pd.DataFrame(filas)
display(R.round(3))

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for m, d in R.groupby("modelo"):
    ax[0].plot(d.k, d.silhouette, "o-", label=m); ax[1].plot(d.k, d.davies_bouldin, "o-", label=m); ax[2].plot(d.k, d.estabilidad_ARI, "o-", label=m)
for a, t in zip(ax, ["Silhouette ↑", "Davies-Bouldin ↓", "Estabilidad (ARI) ↑"]): a.set_title(t); a.set_xlabel("k"); a.legend()
plt.tight_layout(); plt.show()

### Elección del modelo final
Criterio (documentarlo en el pitch): máxima **silhouette + estabilidad**, penalizando grupos con <3 unidades (un grupo con una sola unidad suele ser el Nido o un atípico; es válido, pero conviene que sea explicable).
Puedes sobrescribir con `MODELO_FINAL` / `K_FINAL` si prefieres un k más interpretable para el jurado (3 o 4 grupos suele contar mejor la historia de riesgo).

In [ ]:
cand = R[R.min_tam_grupo >= 3].copy()
if cand.empty: cand = R[R.min_tam_grupo >= 2].copy()     # pocas unidades: relaja el tamaño mínimo
if cand.empty: cand = R.copy()
cand["score"] = cand.silhouette.rank() + cand.estabilidad_ARI.rank() - 0.5 * cand.davies_bouldin.rank()
best = cand.sort_values("score", ascending=False).iloc[0]
MODELO_FINAL, K_FINAL = best.modelo, int(best.k)
# MODELO_FINAL, K_FINAL = "KMeans", 4        # <- descomenta para forzar
print("Modelo elegido:", MODELO_FINAL, "| k =", K_FINAL)
F["grupo_raw"] = ajustar(MODELO_FINAL, K_FINAL, Z)

## 7. Interpretación: ¿qué es cada grupo?
Los grupos se **ordenan por un índice de actividad/peligrosidad relativa** calculado solo para etiquetarlos (promedio de z-scores de frecuencia, energía, magnitud máxima y % superficial). Es descriptivo: el modelo no lo usa para agrupar.

In [ ]:
IND = [c for c in ["sismos_x_anio", "log_energia", "mag_max", "pct_m4_mas", "pct_superficial"] if c in X]
zs = (X[IND] - X[IND].mean()) / X[IND].std()
F["indice_actividad"] = zs.mean(axis=1)

orden = F.groupby("grupo_raw").indice_actividad.mean().sort_values().index
nombres = {1: ["Bajo"], 2: ["Bajo", "Alto"], 3: ["Bajo", "Medio", "Alto"],
           4: ["Bajo", "Moderado", "Alto", "Muy alto"], 5: ["Muy bajo", "Bajo", "Medio", "Alto", "Muy alto"]}
etq = nombres.get(K_FINAL, [f"Nivel {i+1}" for i in range(K_FINAL)])
mapa_g = {g: f"G{i+1} · {etq[i]}" for i, g in enumerate(orden)}
F["grupo"] = F.grupo_raw.map(mapa_g)

perfil = F.groupby("grupo")[COLS_MODELO + ["dist_nido_km"]].median().round(2) if "dist_nido_km" not in COLS_MODELO else F.groupby("grupo")[COLS_MODELO].median().round(2)
perfil.insert(0, "n_unidades", F.groupby("grupo").size())
display(perfil.T)

print("\nUnidades por grupo:")
for g, d in F.sort_values("indice_actividad", ascending=False).groupby("grupo"):
    print(f"  {g} ({len(d)}): {', '.join(d.index[:12])}{' ...' if len(d) > 12 else ''}")

### Importancia de variables (sin modelos supervisados)
Se mide cuánto **separa cada variable a los grupos** con ANOVA de Kruskal-Wallis y tamaño de efecto (η² ≈ H/(n−1)), y se complementa con las cargas del PCA. Así la explicación sigue siendo 100 % no supervisada.

In [ ]:
imp = []
for c in COLS_MODELO:
    muestras = [d[c].values for _, d in F.groupby("grupo")]
    if all(np.std(m) == 0 for m in muestras): continue
    H, p = stats.kruskal(*muestras)
    imp.append((c, H / (len(F) - 1), p))
imp = pd.DataFrame(imp, columns=["variable", "eta2", "p_valor"]).sort_values("eta2", ascending=False)
display(imp.round(4).head(12))

ax = imp.head(12).iloc[::-1].plot.barh(x="variable", y="eta2", legend=False, color="#8e44ad", figsize=(7, 4.5))
ax.set_title("Variables que más separan los grupos (η²)"); plt.tight_layout(); plt.show()

carg = pd.DataFrame(PCA(n_components=2, random_state=RANDOM_STATE).fit(Xs).components_.T, index=COLS_MODELO, columns=["PC1", "PC2"])
display(carg.round(2).sort_values("PC1", key=abs, ascending=False).head(8))

## 8. Visualización territorial (reproducible)

In [ ]:
# 8.1 Espacio de componentes principales
fig, ax = plt.subplots(figsize=(8, 6))
for g, d in F.groupby("grupo"):
    i = [F.index.get_loc(u) for u in d.index]
    ax.scatter(Z2[i, 0], Z2[i, 1], s=60, label=g, alpha=.8)
for u in F.nlargest(6, "n_sismos").index:
    i = F.index.get_loc(u); ax.annotate(u.title(), Z2[i], fontsize=8)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.legend(title="Grupo"); ax.set_title("Grupos en el plano PCA")
plt.tight_layout(); plt.show()

# 8.2 Mapa estático
fig, ax = plt.subplots(figsize=(7.5, 8))
ax.scatter(df.lon, df.lat, s=1, c="lightgray", alpha=.4)
for g, d in F.groupby("grupo"):
    ax.scatter(d.lon_centro, d.lat_centro, s=40 + 4 * d.sismos_x_anio.clip(upper=100), alpha=.85, label=g, edgecolor="k", linewidth=.4)
ax.scatter(NIDO_LON, NIDO_LAT, marker="*", s=300, c="red", label="Nido de Bucaramanga (ref.)")
ax.set_xlabel("Longitud"); ax.set_ylabel("Latitud"); ax.legend(fontsize=8); ax.set_title("Grupos territoriales de comportamiento sísmico")
plt.tight_layout(); plt.show()

In [ ]:
# 8.3 Mapa interactivo (se guarda como HTML para el pitch / app)
import os, plotly.express as px
os.makedirs("app", exist_ok=True)
Fm = F.reset_index().rename(columns={"unidad": "unidad"})
Fm["tam"] = Fm.sismos_x_anio.clip(upper=Fm.sismos_x_anio.quantile(.95)) + 1
fn = getattr(px, "scatter_map", None) or px.scatter_mapbox
kw = dict(lat="lat_centro", lon="lon_centro", color="grupo", size="tam", hover_name="unidad", size_max=28, zoom=7.2,
          hover_data={"n_sismos": True, "mag_max": ":.1f", "prof_media": ":.0f", "indice_actividad": ":.2f", "tam": False},
          title="Riesgo sísmico relativo por unidad territorial — Santander",
          category_orders={"grupo": sorted(F.grupo.unique())})
fig = fn(Fm, **kw)
fig.update_layout(map_style="open-street-map") if fn.__name__ == "scatter_map" else fig.update_layout(mapbox_style="open-street-map")
fig.write_html("app/mapa_grupos.html"); fig.show()

## 9. Exportar resultados (entregables 1, 4 y 5)

In [ ]:
os.makedirs("datos/processed", exist_ok=True); os.makedirs("modelos/no_supervisado", exist_ok=True)
F.drop(columns=["grupo_raw"]).reset_index().to_csv("datos/processed/dataset_limpio.csv", index=False)
F[["grupo", "indice_actividad"]].reset_index().to_csv("modelos/no_supervisado/asignacion_grupos.csv", index=False)
perfil.to_csv("modelos/no_supervisado/perfil_grupos.csv"); R.to_csv("modelos/no_supervisado/comparacion_modelos.csv", index=False)
imp.to_csv("modelos/no_supervisado/importancia_variables.csv", index=False)

# artefactos para que el agente RL / la app reutilicen el clustering
import joblib
joblib.dump({"modelo": MODELO_FINAL, "k": K_FINAL, "columnas": COLS_MODELO, "cols_log": COLS_LOG,
             "grupo_por_unidad": F.grupo.to_dict()}, "modelos/no_supervisado/artefactos.joblib")
print("Listo. Mejor modelo:", MODELO_FINAL, "k =", K_FINAL)
print(R[(R.modelo == MODELO_FINAL) & (R.k == K_FINAL)].round(3).to_string(index=False))

## 10. Cómo contarlo en el pitch (y limitaciones honestas)
**Mensaje:** "Sin etiquetas previas, el modelo descubrió *N* grupos de municipios con comportamiento sísmico distinto; el grupo de mayor actividad concentra …; la variable que más los separa es …; esto sirve para priorizar dónde mirar primero."

**Limitaciones que conviene decir tú antes que el jurado:**
- La sismicidad mide *amenaza*, no *riesgo*: el riesgo también depende de población, vivienda y vulnerabilidad (por eso conviene sumar DANE en `RUTA_TERRITORIAL`).
- Municipios con pocos eventos tienen estadísticas inestables (umbral `MIN_EVENTOS`).
- La cobertura de la red cambió con los años (sesgo de detección en magnitudes pequeñas): prueba `MAG_MIN = 1.5` y comprueba que los grupos se mantienen.
- El etiquetado Bajo/Medio/Alto es interpretativo; el modelo no sabe de "riesgo", solo de similitud.

**Para el agente RL (Avanzado):** el grupo y el `indice_actividad` pueden entrar como parte del *estado* de cada zona.